# Employment Prediction - Data Mining Project

## Objective
Predict whether a developer will be employed based on their skills, experience, and background from Stack Overflow developer survey data.

## Dataset Overview
- **Total Samples**: 73,462 developers
- **Original Features**: 15 columns
- **Final ML Features**: 21 optimized features
- **Target**: Employment status (Employed/Unemployed)

## Key Innovation: HaveWorkedWith Column Processing
We transform the semicolon-separated technology string into **9 essential technology features** from 63 categorized technologies:

### Technology Skill Families (63 technologies in 4 categories):
- **Programming (22)**: Python, Java, JavaScript, C++, C#, TypeScript, etc.
- **Web (17)**: HTML/CSS, React.js, Angular, Vue.js, Node.js, etc.
- **Database (13)**: MySQL, PostgreSQL, MongoDB, Redis, etc.
- **Cloud/DevOps (11)**: AWS, Azure, Docker, Kubernetes, Git, etc.

### 4 Skill Family Scores (Percentage-based):
1. **Programming_Score** - % of programming languages known (0-100%)
2. **Web_Score** - % of web technologies known (0-100%)
3. **Database_Score** - % of databases known (0-100%)
4. **CloudDevOps_Score** - % of cloud/devops tools known (0-100%)

### 5 Skill Family Binary Flags & Derived Features:
5. **Has_Programming** - Any programming language? (Yes/No)
6. **Has_Web** - Any web technology? (Yes/No)
7. **Has_Database** - Any database? (Yes/No)  
8. **Has_CloudDevOps** - Any cloud/devops tool? (Yes/No)
9. **Skill_Breadth** - Number of skill families covered (0-4)
10. **Is_FullStack** - Can build complete applications? (Programming + Web + Database)

## Complete Feature Set (21 Features):
### Technology Features (10)
- 4 percentage scores + 4 binary flags + 2 derived features

### Demographics (6)
- Gender (one-hot: Man/Woman/NonBinary), Age (IsYoung), Accessibility, Mental Health

### Professional (5)
- ComputerSkills, Education, Developer role, Professional experience, Salary info

This approach captures both the **depth** (percentage scores) and **breadth** (binary flags) of technical skills while including important demographic and professional factors.

## 1. Data Loading and Initial Inspection {#1-data-loading}

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src directory to path for importing custom modules
sys.path.append('../src')

# Import updated preprocessing module
from preprocessing_clean import (
    main_preprocessing_pipeline,
    SKILL_FAMILIES
)

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print("✅ Updated preprocessing module loaded!")
print(f"✅ Defined {len(SKILL_FAMILIES)} skill families with {sum(len(techs) for techs in SKILL_FAMILIES.values())} total technologies:")

# Display skill families
for family, technologies in SKILL_FAMILIES.items():
    print(f"   • {family} ({len(technologies)} technologies): {', '.join(technologies[:5])}{'...' if len(technologies) > 5 else ''}")

In [ ]:
# Load the processed datasets (using the clean, optimized datasets)
print("Loading preprocessed datasets...")

try:
    # Load the processed datasets
    X_train = pd.read_csv('../data/processed/X_train.csv')
    X_test = pd.read_csv('../data/processed/X_test.csv')
    y_train = pd.read_csv('../data/processed/y_train.csv')
    y_test = pd.read_csv('../data/processed/y_test.csv')
    df_clean = pd.read_csv('../data/processed/preprocessed_data_clean.csv')
    
    print("✅ All preprocessed datasets loaded successfully!")
    print("\n📊 Dataset Dimensions:")
    print(f"   • Training features: {X_train.shape}")
    print(f"   • Test features: {X_test.shape}")
    print(f"   • Training targets: {y_train.shape}")
    print(f"   • Test targets: {y_test.shape}")
    print(f"   • Clean dataset: {df_clean.shape}")
    
    print(f"\n🎯 Target Distribution:")
    target_dist = y_train.iloc[:, 0].value_counts(normalize=True) * 100
    print(f"   • Employed (1): {target_dist[1]:.1f}%")
    print(f"   • Unemployed (0): {target_dist[0]:.1f}%")
    
    print(f"\n?️ Feature Categories ({len(X_train.columns)} total features):")
    tech_features = [f for f in X_train.columns if any(x in f for x in ['Score', 'Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps', 'Skill_', 'Is_FullStack'])]
    gender_features = [f for f in X_train.columns if f.startswith('Gender_')]
    demographic_features = [f for f in X_train.columns if f in ['IsYoung', 'HasAccessibilityNeeds', 'HasMentalHealthConcerns']]
    professional_features = [f for f in X_train.columns if f in ['ComputerSkills', 'EducationLevel_Numeric', 'IsDeveloper', 'HasProfessionalExperience', 'HasSalaryInfo']]
    
    print(f"   • Technology features: {len(tech_features)}")
    print(f"   • Gender features: {len(gender_features)}")
    print(f"   • Demographic features: {len(demographic_features)}")
    print(f"   • Professional features: {len(professional_features)}")
    
    # Also load original data for comparison
    df_original = pd.read_csv('../data/raw/stackoverflow_with_nulls.csv')
    print(f"\n📈 Transformation Summary:")
    print(f"   • Original dataset: {df_original.shape}")
    print(f"   • Clean dataset: {df_clean.shape}")
    print(f"   • ML feature set: {X_train.shape[1]} features")
    print(f"   • Feature engineering: {df_original.shape[1]} → {X_train.shape[1]} features")
    
except FileNotFoundError as e:
    print(f"❌ Dataset file not found: {e}")
    print("📁 Please ensure all processed datasets are in the '../data/processed/' directory")
    print("🔄 Run the preprocessing pipeline first to generate the datasets")
    df_clean = None

## 2. Dataset Overview and Feature Analysis

In [ ]:
# Display complete feature list with categorization
if X_train is not None:
    print("🎯 COMPLETE FEATURE LIST (21 Features)")
    print("=" * 50)
    
    # Categorize all features
    feature_categories = {
        'Technology': [f for f in X_train.columns if any(x in f for x in ['Score', 'Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps', 'Skill_', 'Is_FullStack'])],
        'Gender': [f for f in X_train.columns if f.startswith('Gender_')],
        'Demographics': [f for f in X_train.columns if f in ['IsYoung', 'HasAccessibilityNeeds', 'HasMentalHealthConcerns']],
        'Professional': [f for f in X_train.columns if f in ['ComputerSkills', 'EducationLevel_Numeric', 'IsDeveloper', 'HasProfessionalExperience', 'HasSalaryInfo']]
    }
    
    for category, features in feature_categories.items():
        print(f"\n{category} Features ({len(features)}):")
        for i, feature in enumerate(features, 1):
            print(f"   {i:2d}. {feature}")
    
    # Check for any uncategorized features
    all_categorized = []
    for features in feature_categories.values():
        all_categorized.extend(features)
    
    uncategorized = [f for f in X_train.columns if f not in all_categorized]
    if uncategorized:
        print(f"\n⚠️ Uncategorized Features ({len(uncategorized)}):")
        for feature in uncategorized:
            print(f"   • {feature}")
    else:
        print(f"\n✅ All {len(X_train.columns)} features properly categorized!")

## 3. Technology Skills Analysis

In [ ]:
# Analyze technology skill scores and distribution
if df_clean is not None:
    # Technology score columns
    score_columns = ['Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score']
    binary_columns = ['Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps']
    
    print("📊 TECHNOLOGY SKILLS ANALYSIS")
    print("=" * 40)
    
    # Score distributions
    print("\n1. SKILL FAMILY SCORE STATISTICS:")
    score_stats = df_clean[score_columns].describe()
    print(score_stats.round(2))
    
    # Binary skill adoption rates
    print("\n2. SKILL FAMILY ADOPTION RATES:")
    for col in binary_columns:
        adoption_rate = df_clean[col].mean() * 100
        count = df_clean[col].sum()
        print(f"   • {col.replace('Has_', '')}: {adoption_rate:.1f}% ({count:,} developers)")
    
    # Skill breadth analysis
    print("\n3. SKILL BREADTH ANALYSIS:")
    skill_breadth_dist = df_clean['Skill_Breadth'].value_counts().sort_index()
    for breadth, count in skill_breadth_dist.items():
        percentage = (count / len(df_clean)) * 100
        print(f"   • {breadth} skill families: {count:,} developers ({percentage:.1f}%)")
    
    # Full-stack developers
    fullstack_count = df_clean['Is_FullStack'].sum()
    fullstack_rate = (fullstack_count / len(df_clean)) * 100
    print(f"\n4. FULL-STACK DEVELOPERS:")
    print(f"   • Full-stack developers: {fullstack_count:,} ({fullstack_rate:.1f}%)")
    print(f"   • Non-full-stack: {len(df_clean) - fullstack_count:,} ({100-fullstack_rate:.1f}%)")

In [ ]:
# Visualize skill family distributions
if df_clean is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Technology Skill Family Distributions', fontsize=16, fontweight='bold')
    
    # Plot score distributions
    score_cols = ['Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score']
    colors = ['skyblue', 'lightgreen', 'coral', 'plum']
    
    for i, (col, color) in enumerate(zip(score_cols, colors)):
        row, col_idx = i // 2, i % 2
        ax = axes[row, col_idx]
        
        # Plot histogram
        ax.hist(df_clean[col], bins=30, alpha=0.7, color=color, edgecolor='black')
        ax.set_title(f'{col.replace("_Score", "")} Skill Scores', fontweight='bold')
        ax.set_xlabel('Score (0-100%)')
        ax.set_ylabel('Number of Developers')
        ax.grid(True, alpha=0.3)
        
        # Add statistics text
        mean_val = df_clean[col].mean()
        median_val = df_clean[col].median()
        ax.axvline(mean_val, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_val:.1f}%')
        ax.axvline(median_val, color='orange', linestyle='--', alpha=0.8, label=f'Median: {median_val:.1f}%')
        ax.legend()
    
    plt.tight_layout()
    plt.show()

## 4. Demographic Analysis

In [ ]:
# Analyze demographic characteristics
if df_clean is not None:
    print("👥 DEMOGRAPHIC ANALYSIS")
    print("=" * 40)
    
    # Age analysis
    print("\n1. AGE DISTRIBUTION:")
    young_count = df_clean['IsYoung'].sum()
    young_rate = (young_count / len(df_clean)) * 100
    print(f"   • Young developers (≤30): {young_count:,} ({young_rate:.1f}%)")
    print(f"   • Older developers (>30): {len(df_clean) - young_count:,} ({100-young_rate:.1f}%)")
    
    # Gender analysis
    print("\n2. GENDER DISTRIBUTION:")
    male_count = df_clean['Gender_Male'].sum()
    female_count = df_clean['Gender_Female'].sum()
    nonbinary_count = df_clean['Gender_NonBinary'].sum()
    total = len(df_clean)
    
    print(f"   • Male: {male_count:,} ({(male_count/total)*100:.1f}%)")
    print(f"   • Female: {female_count:,} ({(female_count/total)*100:.1f}%)")
    print(f"   • Non-binary: {nonbinary_count:,} ({(nonbinary_count/total)*100:.1f}%)")
    
    # Education analysis
    print("\n3. EDUCATION LEVEL:")
    ed_stats = df_clean['EducationLevel_Numeric'].describe()
    print(f"   • Mean education level: {ed_stats['mean']:.2f}")
    print(f"   • Median education level: {ed_stats['50%']:.2f}")
    print(f"   • Range: {ed_stats['min']:.0f} - {ed_stats['max']:.0f}")
    
    # Professional characteristics
    print("\n4. PROFESSIONAL CHARACTERISTICS:")
    prof_exp_count = df_clean['HasProfessionalExperience'].sum()
    prof_exp_rate = (prof_exp_count / len(df_clean)) * 100
    
    accessibility_count = df_clean['HasAccessibilityNeeds'].sum()
    accessibility_rate = (accessibility_count / len(df_clean)) * 100
    
    mental_health_count = df_clean['HasMentalHealthConcerns'].sum()
    mental_health_rate = (mental_health_count / len(df_clean)) * 100
    
    developer_count = df_clean['IsDeveloper'].sum()
    developer_rate = (developer_count / len(df_clean)) * 100
    
    print(f"   • Professional experience: {prof_exp_count:,} ({prof_exp_rate:.1f}%)")
    print(f"   • Accessibility needs: {accessibility_count:,} ({accessibility_rate:.1f}%)")
    print(f"   • Mental health concerns: {mental_health_count:,} ({mental_health_rate:.1f}%)")
    print(f"   • Primary developers: {developer_count:,} ({developer_rate:.1f}%)")

In [ ]:
# Visualize demographic distributions
if df_clean is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Demographic Characteristics Distribution', fontsize=16, fontweight='bold')
    
    # Age distribution
    age_data = ['Young (≤30)', 'Older (>30)']
    age_counts = [df_clean['IsYoung'].sum(), len(df_clean) - df_clean['IsYoung'].sum()]
    axes[0, 0].pie(age_counts, labels=age_data, autopct='%1.1f%%', startangle=90, colors=['lightblue', 'lightcoral'])
    axes[0, 0].set_title('Age Distribution', fontweight='bold')
    
    # Gender distribution
    gender_data = ['Male', 'Female', 'Non-Binary']
    gender_counts = [df_clean['Gender_Male'].sum(), df_clean['Gender_Female'].sum(), df_clean['Gender_NonBinary'].sum()]
    axes[0, 1].pie(gender_counts, labels=gender_data, autopct='%1.1f%%', startangle=90, colors=['skyblue', 'pink', 'lightgreen'])
    axes[0, 1].set_title('Gender Distribution', fontweight='bold')
    
    # Education level histogram
    axes[1, 0].hist(df_clean['EducationLevel_Numeric'], bins=20, alpha=0.7, color='gold', edgecolor='black')
    axes[1, 0].set_title('Education Level Distribution', fontweight='bold')
    axes[1, 0].set_xlabel('Education Level (Numeric)')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Professional characteristics
    prof_chars = ['Professional\nExperience', 'Accessibility\nNeeds', 'Mental Health\nConcerns', 'Primary\nDeveloper']
    prof_counts = [
        df_clean['HasProfessionalExperience'].sum(),
        df_clean['HasAccessibilityNeeds'].sum(),
        df_clean['HasMentalHealthConcerns'].sum(),
        df_clean['IsDeveloper'].sum()
    ]
    prof_percentages = [(count / len(df_clean)) * 100 for count in prof_counts]
    
    bars = axes[1, 1].bar(prof_chars, prof_percentages, color=['lightgreen', 'orange', 'lightcoral', 'skyblue'])
    axes[1, 1].set_title('Professional Characteristics', fontweight='bold')
    axes[1, 1].set_ylabel('Percentage (%)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Add percentage labels on bars
    for bar, percentage in zip(bars, prof_percentages):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                       f'{percentage:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 5. Employment Status Analysis

In [ ]:
# Analyze employment status (target variable)
if y_train is not None and y_test is not None:
    # Combine target variables for full analysis
    y_full = pd.concat([y_train, y_test], ignore_index=True)
    
    print("💼 EMPLOYMENT STATUS ANALYSIS")
    print("=" * 40)
    
    # Overall employment distribution
    employment_dist = y_full.value_counts()
    print("\n1. OVERALL EMPLOYMENT DISTRIBUTION:")
    for status, count in employment_dist.items():
        percentage = (count / len(y_full)) * 100
        print(f"   • {status}: {count:,} ({percentage:.1f}%)")
    
    # Training vs Test distribution
    print("\n2. TRAINING vs TEST DISTRIBUTION:")
    train_dist = y_train.value_counts()
    test_dist = y_test.value_counts()
    
    print("   Training Set:")
    for status, count in train_dist.items():
        percentage = (count / len(y_train)) * 100
        print(f"     • {status}: {count:,} ({percentage:.1f}%)")
    
    print("   Test Set:")
    for status, count in test_dist.items():
        percentage = (count / len(y_test)) * 100
        print(f"     • {status}: {count:,} ({percentage:.1f}%)")
    
    # Check class balance
    employed_ratio = y_full.value_counts()['Employed'] / len(y_full)
    print(f"\n3. CLASS BALANCE:")
    print(f"   • Employment ratio: {employed_ratio:.3f}")
    if 0.4 <= employed_ratio <= 0.6:
        print("   • ✅ Well-balanced classes")
    else:
        print("   • ⚠️ Imbalanced classes - consider balancing techniques")

In [ ]:
# Visualize employment status distribution
if y_train is not None and y_test is not None:
    y_full = pd.concat([y_train, y_test], ignore_index=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Employment Status Distribution', fontsize=16, fontweight='bold')
    
    # Overall distribution pie chart
    employment_counts = y_full.value_counts()
    colors = ['lightgreen' if status == 'Employed' else 'lightcoral' for status in employment_counts.index]
    
    axes[0].pie(employment_counts.values, labels=employment_counts.index, autopct='%1.1f%%', 
                startangle=90, colors=colors)
    axes[0].set_title('Overall Employment Distribution', fontweight='bold')
    
    # Training vs Test comparison
    train_counts = y_train.value_counts()
    test_counts = y_test.value_counts()
    
    categories = ['Employed', 'Unemployed']
    train_values = [train_counts.get(cat, 0) for cat in categories]
    test_values = [test_counts.get(cat, 0) for cat in categories]
    
    x = range(len(categories))
    width = 0.35
    
    bars1 = axes[1].bar([i - width/2 for i in x], train_values, width, label='Training', alpha=0.8, color='skyblue')
    bars2 = axes[1].bar([i + width/2 for i in x], test_values, width, label='Test', alpha=0.8, color='orange')
    
    axes[1].set_title('Training vs Test Set Distribution', fontweight='bold')
    axes[1].set_xlabel('Employment Status')
    axes[1].set_ylabel('Count')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(categories)
    axes[1].legend()
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2., height + 100,
                        f'{int(height):,}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 6. Feature Correlation Analysis

In [ ]:
# Analyze correlations between features
if X_train is not None and df_clean is not None:
    print("🔗 FEATURE CORRELATION ANALYSIS")
    print("=" * 40)
    
    # Calculate correlation matrix
    correlation_matrix = X_train.corr()
    
    # Find highly correlated feature pairs (excluding self-correlation)
    print("\n1. HIGHLY CORRELATED FEATURE PAIRS (|r| > 0.5):")
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.5:
                feature1 = correlation_matrix.columns[i]
                feature2 = correlation_matrix.columns[j]
                high_corr_pairs.append((feature1, feature2, corr_val))
                print(f"   • {feature1} ↔ {feature2}: {corr_val:.3f}")
    
    if not high_corr_pairs:
        print("   • No highly correlated pairs found (good for multicollinearity)")
    
    # Analyze correlations within feature groups
    print("\n2. FEATURE GROUP CORRELATIONS:")
    
    # Technology features
    tech_features = ['Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score',
                    'Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps',
                    'Skill_Breadth', 'Is_FullStack']
    
    # Gender features
    gender_features = ['Gender_Male', 'Gender_Female', 'Gender_NonBinary']
    
    # Professional features
    prof_features = ['HasProfessionalExperience', 'IsDeveloper', 'EducationLevel_Numeric',
                    'HasAccessibilityNeeds', 'HasMentalHealthConcerns']
    
    # Demographic features
    demo_features = ['IsYoung'] + gender_features
    
    feature_groups = {
        'Technology Skills': tech_features,
        'Gender': gender_features,
        'Professional': prof_features,
        'Demographics': demo_features
    }
    
    for group_name, features in feature_groups.items():
        available_features = [f for f in features if f in X_train.columns]
        if len(available_features) > 1:
            group_corr = X_train[available_features].corr()
            avg_corr = group_corr.abs().mean().mean()
            print(f"   • {group_name}: Average |correlation| = {avg_corr:.3f}")
    
    # Show top correlated features with each technology skill
    print("\n3. TOP CORRELATIONS WITH SKILL SCORES:")
    skill_scores = ['Programming_Score', 'Web_Score', 'Database_Score', 'CloudDevOps_Score']
    for skill in skill_scores:
        if skill in X_train.columns:
            skill_corr = X_train.corr()[skill].abs().sort_values(ascending=False)
            skill_corr = skill_corr[skill_corr.index != skill]  # Remove self-correlation
            print(f"   • {skill}:")
            for i, (feature, corr_val) in enumerate(skill_corr.head(3).items()):
                print(f"     {i+1}. {feature}: {corr_val:.3f}")

In [ ]:
# Visualize correlation matrix
if X_train is not None:
    plt.figure(figsize=(16, 14))
    
    # Calculate correlation matrix
    correlation_matrix = X_train.corr()
    
    # Create heatmap
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))  # Mask upper triangle
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.2f')
    
    plt.title('Feature Correlation Matrix\n(Lower Triangle Only)', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Correlation matrix shows relationships between all {len(X_train.columns)} features")
    print("🔴 Strong positive correlations (red) | 🔵 Strong negative correlations (blue)")
    print("⚪ Weak/no correlations (white)")

## 7. Employment vs Features Analysis

In [ ]:
# Analyze relationship between features and employment status
if X_train is not None and y_train is not None and df_clean is not None:
    # Combine training data for analysis
    df_analysis = X_train.copy()
    df_analysis['Employment_Status'] = y_train.values
    
    print("💼 EMPLOYMENT vs FEATURES ANALYSIS")
    print("=" * 40)
    
    # Analyze employment rates by different characteristics
    print("\n1. EMPLOYMENT RATES BY CHARACTERISTICS:")
    
    # Technology skills impact
    tech_binary_features = ['Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps', 'Is_FullStack']
    for feature in tech_binary_features:
        if feature in df_analysis.columns:
            employment_rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            print(f"   • {feature.replace('Has_', '').replace('Is_', '')}: {employment_rate:.1%} employment rate")
    
    # Demographic characteristics impact
    demo_features = ['IsYoung', 'Gender_Male', 'Gender_Female', 'Gender_NonBinary']
    for feature in demo_features:
        if feature in df_analysis.columns:
            employment_rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            print(f"   • {feature.replace('Gender_', '').replace('Is', '')}: {employment_rate:.1%} employment rate")
    
    # Professional characteristics impact
    prof_features = ['HasProfessionalExperience', 'IsDeveloper', 'HasAccessibilityNeeds', 'HasMentalHealthConcerns']
    for feature in prof_features:
        if feature in df_analysis.columns:
            employment_rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            print(f"   • {feature.replace('Has', '').replace('Is', '')}: {employment_rate:.1%} employment rate")
    
    # Skill breadth analysis
    print("\n2. EMPLOYMENT BY SKILL BREADTH:")
    if 'Skill_Breadth' in df_analysis.columns:
        breadth_employment = df_analysis.groupby('Skill_Breadth')['Employment_Status'].apply(
            lambda x: (x == 'Employed').mean()
        ).sort_index()
        for breadth, rate in breadth_employment.items():
            count = (df_analysis['Skill_Breadth'] == breadth).sum()
            print(f"   • {breadth} skill families: {rate:.1%} employment rate ({count:,} developers)")
    
    # Education level analysis
    print("\n3. EMPLOYMENT BY EDUCATION LEVEL:")
    if 'EducationLevel_Numeric' in df_analysis.columns:
        # Group education levels for analysis
        df_analysis['Education_Group'] = pd.cut(df_analysis['EducationLevel_Numeric'], 
                                               bins=[0, 2, 4, 6, float('inf')], 
                                               labels=['Low (0-2)', 'Medium (3-4)', 'High (5-6)', 'Very High (7+)'])
        education_employment = df_analysis.groupby('Education_Group')['Employment_Status'].apply(
            lambda x: (x == 'Employed').mean()
        )
        for edu_level, rate in education_employment.items():
            count = (df_analysis['Education_Group'] == edu_level).sum()
            print(f"   • {edu_level}: {rate:.1%} employment rate ({count:,} developers)")
            
    # Overall insights
    overall_employment_rate = (df_analysis['Employment_Status'] == 'Employed').mean()
    print(f"\n4. BASELINE EMPLOYMENT RATE: {overall_employment_rate:.1%}")
    print("   Features with higher employment rates may be strong predictors!")

In [ ]:
# Visualize employment rates by key characteristics
if X_train is not None and y_train is not None:
    df_analysis = X_train.copy()
    df_analysis['Employment_Status'] = y_train.values
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Employment Rates by Key Characteristics', fontsize=16, fontweight='bold')
    
    # Technology skills employment rates
    tech_features = ['Has_Programming', 'Has_Web', 'Has_Database', 'Has_CloudDevOps', 'Is_FullStack']
    tech_rates = []
    tech_labels = []
    
    for feature in tech_features:
        if feature in df_analysis.columns:
            rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            tech_rates.append(rate * 100)
            tech_labels.append(feature.replace('Has_', '').replace('Is_', ''))
    
    bars1 = axes[0, 0].bar(tech_labels, tech_rates, color='lightblue', alpha=0.8)
    axes[0, 0].set_title('Employment Rates by Technology Skills', fontweight='bold')
    axes[0, 0].set_ylabel('Employment Rate (%)')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Add percentage labels
    for bar, rate in zip(bars1, tech_rates):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                       f'{rate:.1f}%', ha='center', va='bottom')
    
    # Skill breadth employment rates
    if 'Skill_Breadth' in df_analysis.columns:
        breadth_employment = df_analysis.groupby('Skill_Breadth')['Employment_Status'].apply(
            lambda x: (x == 'Employed').mean() * 100
        ).sort_index()
        
        bars2 = axes[0, 1].bar(breadth_employment.index, breadth_employment.values, 
                              color='lightgreen', alpha=0.8)
        axes[0, 1].set_title('Employment Rate by Skill Breadth', fontweight='bold')
        axes[0, 1].set_xlabel('Number of Skill Families')
        axes[0, 1].set_ylabel('Employment Rate (%)')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Add percentage labels
        for bar, rate in zip(bars2, breadth_employment.values):
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                           f'{rate:.1f}%', ha='center', va='bottom')
    
    # Demographics employment rates
    demo_features = ['IsYoung', 'Gender_Male', 'Gender_Female', 'Gender_NonBinary']
    demo_rates = []
    demo_labels = []
    
    for feature in demo_features:
        if feature in df_analysis.columns:
            rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            demo_rates.append(rate * 100)
            demo_labels.append(feature.replace('Gender_', '').replace('Is', ''))
    
    bars3 = axes[1, 0].bar(demo_labels, demo_rates, color='coral', alpha=0.8)
    axes[1, 0].set_title('Employment Rates by Demographics', fontweight='bold')
    axes[1, 0].set_ylabel('Employment Rate (%)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Add percentage labels
    for bar, rate in zip(bars3, demo_rates):
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                       f'{rate:.1f}%', ha='center', va='bottom')
    
    # Professional characteristics employment rates
    prof_features = ['HasProfessionalExperience', 'IsDeveloper', 'HasAccessibilityNeeds', 'HasMentalHealthConcerns']
    prof_rates = []
    prof_labels = []
    
    for feature in prof_features:
        if feature in df_analysis.columns:
            rate = df_analysis[df_analysis[feature] == 1]['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0).mean()
            prof_rates.append(rate * 100)
            prof_labels.append(feature.replace('Has', '').replace('Is', ''))
    
    bars4 = axes[1, 1].bar(prof_labels, prof_rates, color='gold', alpha=0.8)
    axes[1, 1].set_title('Employment Rates by Professional Characteristics', fontweight='bold')
    axes[1, 1].set_ylabel('Employment Rate (%)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
    
    # Add percentage labels
    for bar, rate in zip(bars4, prof_rates):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                       f'{rate:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 8. Key Insights and Conclusions

In [ ]:
# Summary of key insights from EDA
print("🎯 KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 50)

print("\n📊 DATASET CHARACTERISTICS:")
print(f"   • Total samples: {len(df_clean):,} developers")
print(f"   • Training samples: {len(X_train):,} | Test samples: {len(X_test):,}")
print(f"   • Features: {len(X_train.columns)} optimized features")
print(f"   • Target distribution: Balanced classes (Employment ratio ≈ 0.54)")

print("\n🛠️ TECHNOLOGY SKILLS INSIGHTS:")
print("   • Programming skills are most common among developers")
print("   • Full-stack developers represent a significant portion")
print("   • Skill breadth varies widely - from 0 to 4 skill families")
print("   • Technology scores show diverse developer expertise levels")

print("\n👥 DEMOGRAPHIC INSIGHTS:")
print("   • Majority of developers are male (reflecting industry trends)")
print("   • Age distribution shows both young and experienced developers")
print("   • Education levels vary significantly across the dataset")
print("   • Mental health concerns affect a notable portion of developers")

print("\n💼 EMPLOYMENT PATTERNS:")
print("   • Employment rates vary by technology skills")
print("   • Professional experience strongly correlates with employment")
print("   • Skill diversity may impact employment opportunities")
print("   • Demographic factors show varying employment correlations")

print("\n🔍 FEATURE RELATIONSHIPS:")
print("   • Low multicollinearity - good for machine learning models")
print("   • Technology features show expected correlations")
print("   • Gender features are mutually exclusive (as expected)")
print("   • Professional characteristics cluster together")

print("\n📈 MODELING RECOMMENDATIONS:")
print("   • Dataset is well-prepared for machine learning")
print("   • Balanced classes reduce need for special sampling techniques")
print("   • 21 features provide comprehensive developer profile")
print("   • Feature engineering successfully captured skill diversity")
print("   • Ready for model training and evaluation")

print("\n✅ DATA QUALITY:")
print("   • No missing values in final dataset")
print("   • All categorical variables properly encoded")
print("   • Features scaled and ready for ML algorithms")
print("   • Train/test split maintains class balance")

print("\n🚀 NEXT STEPS:")
print("   • Proceed to model training with multiple algorithms")
print("   • Compare model performance using cross-validation")
print("   • Feature importance analysis to identify key predictors")
print("   • Model interpretation and business insights extraction")